In [17]:
import pandas as pd
# chargement du dataset
df = pd.read_csv("dataset_avis.csv")
df.head()

,comment_id,Commentaire,star,date,client,reponse,source,company,ville,maj,date_commande,ecart,clean_comment,qualité produit,service livraison,service client
0,2,"Vente Lacoste Honteuse , article erroné , arti...",1,2021-06-19 00:00:00+00:00,Vanessa L,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,"vente lacoste honteuse , article erroné , arti...",1,0,0
1,6,Annulation de commande après 2 mois d ’ attent...,1,2021-06-18 00:00:00+00:00,aurore regnier,"Bonjour Aurore , Je suis sincèrement désolé d'...",TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,annulation de commande après 2 mois d ’ attent...,0,0,1
2,8,Extrêmement deçu pour mes achats lors la vente...,1,2021-06-18 00:00:00+00:00,Ayna,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,extrêmement deçu pour mes achats lors la vente...,1,0,0
3,9,S'il y'avait une option : ne pas mettre d'étoi...,1,2021-06-18 00:00:00+00:00,linda Ng,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,s'il y'avait une option : ne pas mettre d'étoi...,1,0,0
4,10,ARNAQUE J ’ ai acheté une combinaison blanche ...,1,2021-06-18 00:00:00+00:00,Sarah,NaN,TrustPilot,ShowRoom,NaN,NaN,NaN,NaN,arnaque j ’ ai acheté une combinaison blanche ...,1,0,0


In [3]:
# colonnes cibles(multilabel)
labels = ["qualité produit", "service livraison", "service client"]


In [18]:
# Données texte (features)
X = df["clean_comment"].values
X.shape

(6535,)

In [16]:
# Données cibles
y = df[labels].values
print(y)

[[1 0 0]
 [0 0 1]
 [1 0 0]
 ...
 [0 1 0]
 [1 0 0]
 [1 0 0]]


In [6]:
#TF-IDF vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

X_tfidf = tfidf.fit_transform(X)

In [7]:
#Séparation Train / Test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
# Modèle SVM Multilabel
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

# LinearSVC as base model
svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

# multilable avec OneVsRest
clf = OneVsRestClassifier(svm_model)

In [9]:
#Validation croisée (F1 micro)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_tfidf,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro per fold :", cv_scores)
print("F1 micro average  :", cv_scores.mean())
print("Standard deviation :", cv_scores.std())

F1 micro per fold : [0.64876678 0.76193922 0.7558098  0.76108108 0.60930802]
F1 micro average  : 0.7073809798006359
Standard deviation : 0.06520672032258247


In [10]:
from sklearn.metrics import classification_report

# Train on train set
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

#  Évaluation sur le jeu de test
print(classification_report(y_test, y_pred, target_names=labels))


                   precision    recall  f1-score   support

  qualité produit       0.75      0.80      0.77       717
service livraison       0.75      0.68      0.72       547
   service client       0.46      0.46      0.46       209

        micro avg       0.71      0.71      0.71      1473
        macro avg       0.65      0.65      0.65      1473
     weighted avg       0.71      0.71      0.71      1473
      samples avg       0.70      0.73      0.70      1473



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
